In [ ]:
# =========================
# Fresh Feature Extraction -> CSV
# =========================
!pip -q install librosa soundfile

import os, glob, warnings
import numpy as np
import pandas as pd
import librosa

warnings.filterwarnings("ignore")


DATASET_DIR = "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset"

# 2) Your genre folder names (exactly as in your dataset)
GENRE_FOLDERS = [
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Afrobeat/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Afrobeats/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Amapiano/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Congolese Rumba/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Fuji/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Highlife/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Juju/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Kwaito/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Mbalax/Trimmed",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Soukous/Trimmed",
]

# Optional: map folder name -> canonical label (edit if you want them separate)
CANON_LABEL = {
    "Afrobeat": "Afrobeat",
    "Afrobeats": "Afrobeat",          # merge Afrobeat/Afrobeats (change if you want separate)
    "Amapiano": "Amapiano",
    "Congolese Rumba": "Congolese Rumba",
    "Fuji": "Fuji",
    "Highlife": "Highlife",
    "Juju": "Juju",
    "Kwaito": "Kwaito",
    "Mbalax": "Mbalax",
    "Soukous & Makossa": "Soukous & Makossa",
}

# 3) Audio + feature settings
SR = 22050
DURATION = 30.0     # seconds to load from each file
OFFSET = 0.0
N_MFCC = 20

AUDIO_EXTS = ("*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a", "*.aac", "*.wma")

def stats(prefix, x):
    """Return mean/var for a 1D array."""
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_var":  float(np.var(x)),
    }

def extract_features(y, sr):
    feats = {}

    # Basic time-domain
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y)[0]
    feats.update(stats("zcr", zcr))
    feats.update(stats("rms", rms))

    # Tempo / beat
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    feats["tempo"] = float(tempo)

    # Spectral
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    feats.update(stats("spectral_centroid", centroid))
    feats.update(stats("spectral_bandwidth", bandwidth))
    feats.update(stats("spectral_rolloff", rolloff))
    # contrast has multiple bands
    for i in range(contrast.shape[0]):
        feats.update(stats(f"spectral_contrast_{i+1}", contrast[i]))

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(chroma.shape[0]):
        feats.update(stats(f"chroma_{i+1}", chroma[i]))

    # Mel-spectrogram (log-mel energy stats)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # summarize mel bands
    feats.update(stats("mel_db", mel_db.flatten()))

    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    for i in range(mfcc.shape[0]):
        feats.update(stats(f"mfcc{i+1}", mfcc[i]))

    return feats

rows = []
bad_files = []

for folder in GENRE_FOLDERS:
    genre_dir = os.path.join(DATASET_DIR, folder)
    if not os.path.isdir(genre_dir):
        print("Missing folder:", genre_dir)
        continue

    # collect files in this genre folder
    files = []
    for ext in AUDIO_EXTS:
        files.extend(glob.glob(os.path.join(genre_dir, ext)))

    print(f"{folder}: {len(files)} files")

    for fp in files:
        try:
            y, sr = librosa.load(fp, sr=SR, duration=DURATION, offset=OFFSET, mono=True)

            # ensure non-empty audio
            if y is None or len(y) < 1000:
                bad_files.append((fp, "too_short_or_empty"))
                continue

            feats = extract_features(y, sr)

            rows.append({
                "file_name": os.path.basename(fp),
                "file_path": fp,
                "genre": CANON_LABEL.get(folder, folder),
                **feats
            })
        except Exception as e:
            bad_files.append((fp, str(e)))

df = pd.DataFrame(rows)

# Save outputs
OUT_CSV = "/content/amgc_features_upgraded.csv"
df.to_csv(OUT_CSV, index=False)

print("\nDONE")
print("Feature CSV:", OUT_CSV)
print("Shape:", df.shape)
print("Genre counts:\n", df["genre"].value_counts())




/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Afrobeat/Trimmed: 117 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Afrobeats/Trimmed: 97 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Amapiano/Trimmed: 105 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Congolese Rumba/Trimmed: 106 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Fuji/Trimmed: 111 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Highlife/Trimmed: 103 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Juju/Trimmed: 105 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Kwaito/Trimmed: 105 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Mbalax/Trimmed: 106 files
/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Sou

In [ ]:
import os

out_dir = "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/1. Extracted_Features"
os.makedirs(out_dir, exist_ok=True)

out_csv = os.path.join(out_dir, "amgc_features_upgraded.csv")
df.to_csv(out_csv, index=False)

print("Saved to:", out_csv)


Saved to: /content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/1. Extracted_Features/amgc_features_upgraded.csv


In [ ]:
import pandas as pd
import re

in_path  = "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/1. Extracted_Features/amgc_features_upgraded.csv"
out_path = "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/1. Extracted_Features/amgc_features_upgraded.csv"

df = pd.read_csv(in_path)

# Canonical mapping (exactly as you listed)
MAP = {
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Afrobeat/Trimmed": "Afrobeat",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Afrobeats/Trimmed": "Afrobeats",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Amapiano/Trimmed": "Amapiano",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Congolese Rumba/Trimmed": "Congolese Rumba",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Fuji/Trimmed": "Fuji",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Highlife/Trimmed": "Highlife",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Juju/Trimmed": "juju",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Kwaito/Trimmed": "Kwaito",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Mbalax/Trimmed": "Mbalax",
    "/content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/Soukous & Makossa/Trimmed": "Soukous & Makossa",
}

# Normalize slashes to be safe across OS
df["genre"] = df["genre"].astype(str).str.replace("\\", "/", regex=False).str.strip("/")
norm_map = {k.replace("\\", "/").strip("/"): v for k, v in MAP.items()}

# First try exact mapping
df["genre"] = df["genre"].map(norm_map).fillna(df["genre"])

# Fallback: if still not mapped, extract the genre folder right before "/Trimmed"
df["genre"] = df["genre"].where(
    df["genre"].isin(norm_map.values()),
    df["genre"].astype(str).str.extract(r"/([^/]+)/Trimmed/?$", expand=False)
)

# Final safety: strip whitespace
df["genre"] = df["genre"].astype(str).str.strip()

print("Unique genres:", df["genre"].nunique())
print(df["genre"].value_counts(dropna=False))

df.to_csv(out_path, index=False)
print("Saved:", out_path)



Unique genres: 10
genre
Afrobeat           117
Fuji               111
Soukous            108
Mbalax             106
Congolese Rumba    106
juju               105
Amapiano           105
Highlife           103
Kwaito             103
Afrobeats           97
Name: count, dtype: int64
Saved: /content/drive/MyDrive/African Music Genre classifier Dataset/AMGC Dataset/1. Extracted_Features/amgc_features_upgraded.csv
